In [20]:
import pandas as pd

data = pd.read_csv("aps_failure_training_set.csv"
                   , skiprows=20,
                     na_values="na"
                    )

print("Dataset shape:", data.shape)

print("\n First 5 rows of the dataset:")
print(data.head())
print("\n Column names:")
print(data.columns.tolist())
print("\nTarget distribution:")
print(data['class'].value_counts())


Dataset shape: (60000, 171)

 First 5 rows of the dataset:
  class  aa_000  ab_000        ac_000  ad_000  ae_000  af_000  ag_000  ag_001  \
0   neg   76698     NaN  2.130706e+09   280.0     0.0     0.0     0.0     0.0   
1   neg   33058     NaN  0.000000e+00     NaN     0.0     0.0     0.0     0.0   
2   neg   41040     NaN  2.280000e+02   100.0     0.0     0.0     0.0     0.0   
3   neg      12     0.0  7.000000e+01    66.0     0.0    10.0     0.0     0.0   
4   neg   60874     NaN  1.368000e+03   458.0     0.0     0.0     0.0     0.0   

   ag_002  ...     ee_002    ee_003    ee_004    ee_005    ee_006    ee_007  \
0     0.0  ...  1240520.0  493384.0  721044.0  469792.0  339156.0  157956.0   
1     0.0  ...   421400.0  178064.0  293306.0  245416.0  133654.0   81140.0   
2     0.0  ...   277378.0  159812.0  423992.0  409564.0  320746.0  158022.0   
3     0.0  ...      240.0      46.0      58.0      44.0      10.0       0.0   
4     0.0  ...   622012.0  229790.0  405298.0  347188.0  28

In [21]:
import pandas as pd

missing_count = data.isna().sum()
missing_percentage = data.isna().mean()*100

missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percentage.round(2)
})

missing_summary = missing_summary.sort_values("Missing_Percentage",
                                              ascending=False
                                              )

print("Columns with the most missing values:")
display(missing_summary.head(20))

print(
    "\n Total missing values:",
    int(data.isna().sum().sum())
)

Columns with the most missing values:


,Missing_Count,Missing_Percentage
br_000,49264,82.11
bq_000,48722,81.20
bp_000,47740,79.57
bo_000,46333,77.22
ab_000,46329,77.22
cr_000,46329,77.22
bn_000,44009,73.35
bm_000,39549,65.92
bl_000,27277,45.46
bk_000,23034,38.39



 Total missing values: 850015


In [22]:
from sklearn.model_selection import train_test_split

x=data.drop(columns=['class'])
y=data['class'].eq('pos').astype(int)
# First, reserve 15% for final testing.
X_train_val, X_test, y_train_val, y_test = train_test_split(x, y, test_size=0.15, random_state=42,stratify=y)
# Divide the remaining data into training and validation sets.
x_train, x_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1765, random_state=42,stratify=y_train_val)
# Keep features with at most 70% missing values in the training set.
missing_rate=x_train.isna().mean()
selected_features=missing_rate[missing_rate<=0.7].index

x_train=x_train[selected_features].copy()
x_val=x_val[selected_features].copy()
X_test=X_test[selected_features].copy()

print("training rows ",len(x_train))
print("validation rows ",len(x_val))
print("test rows ",len(X_test))
print("features trained ",len(selected_features))

print("\nPositive cases in each set:")
print("Training:", int(y_train.sum()))
print("Validation:", int(y_val.sum()))
print("Test:", int(y_test.sum()))


training rows  41998
validation rows  9002
test rows  9000
features trained  163

Positive cases in each set:
Training: 700
Validation: 150
Test: 150


In [23]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)


# Create a model that always predicts the most common class
dummy_model = DummyClassifier(strategy="most_frequent")

# "Train" the baseline on the training data
dummy_model.fit(x_train, y_train)

# Predict for the validation set
dummy_predictions = dummy_model.predict(x_val)

# Calculate accuracy, precision, and recall for the dummy model
dummy_accuracy = accuracy_score(y_val, dummy_predictions)
dummy_precision = precision_score(
    y_val, dummy_predictions, zero_division=0
)
dummy_recall = recall_score(
    y_val, dummy_predictions, zero_division=0
)

# Calculate the confusion matrix
dummy_cm = confusion_matrix(
    y_val,
    dummy_predictions,
    labels=[0, 1]
)

tn, fp, fn, tp = dummy_cm.ravel()

# Use the dataset challenge's assumed error costs
dummy_cost = 10 * fp + 500 * fn

print("Dummy Classifier results")
print("------------------------")
print("Accuracy:", round(dummy_accuracy, 4))
print("Precision:", round(dummy_precision, 4))
print("Recall:", round(dummy_recall, 4))

print("\nConfusion matrix:")
print(dummy_cm)

print("\nTrue negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

print("\nChallenge cost:", dummy_cost)

Dummy Classifier results
------------------------
Accuracy: 0.9833
Precision: 0.0
Recall: 0.0

Confusion matrix:
[[8852    0]
 [ 150    0]]

True negatives: 8852
False positives: 0
False negatives: 150
True positives: 0

Challenge cost: 75000


In [24]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Train the pipeline
logistic_model.fit(x_train, y_train)

# Predict class probabilities on the validation set
logistic_probabilities = logistic_model.predict_proba(x_val)[:, 1]

# Convert probabilities into class predictions using the default threshold
logistic_predictions = (logistic_probabilities >= 0.5).astype(int)

print("\nFirst 10 predicted positive-class probabilities:")
print(np.round(logistic_probabilities[:10], 4))

print("\nFirst 10 predicted classes:")
print(logistic_predictions[:10])


First 10 predicted positive-class probabilities:
[0.0485 0.     0.0253 0.0027 0.0288 0.0236 0.1003 0.0567 0.0112 0.0111]

First 10 predicted classes:
[0 0 0 0 0 0 0 0 0 0]


In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Build the preprocessing + Random Forest pipeline
random_forest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        max_depth=15,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1#was used to speed up the training process by using available CPU cores (can remove if not needed) 
        #previosu run time for this cell was 21.8 sec now its 3.5 sec
    ))
])

# Train the model
random_forest_model.fit(x_train, y_train)

# Obtain positive-class probabilities on the validation set
rf_probabilities = random_forest_model.predict_proba(x_val)[:, 1]

# Initial predictions at a threshold of 0.5
rf_predictions = (rf_probabilities >= 0.5).astype(int)

print("\nFirst 10 predicted positive-class probabilities:")
print(np.round(rf_probabilities[:10], 4))

print("\nFirst 10 predicted classes:")
print(rf_predictions[:10])

Random Forest trained successfully.

First 10 predicted positive-class probabilities:
[0.0131 0.0002 0.0058 0.0735 0.0062 0.0001 0.0252 0.0003 0.0006 0.0001]

First 10 predicted classes:
[0 0 0 0 0 0 0 0 0 0]
